# Run the bottle object detector over Johnnie Walker S3 clips

This notebook follows the resumable S3 workflow in `../cnn_classifier/03_classify_s3_clips.ipynb`, but replaces whole-frame CNN classification with the trained `VideoModule` YOLO detector.

For every selected clip it downloads one temporary file, runs detection chronologically, appends frame and clip CSV results immediately, writes an annotated MP4, and removes the temporary source download.

Safe defaults: `run_s3_batch=False` and `max_clips=5`.

## 1. Imports and configuration

In [11]:
import csv
import hashlib
import json
import os
import re
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

import boto3
import cv2
import pandas as pd
import torch
from botocore.exceptions import BotoCoreError, ClientError, TokenRetrievalError

working_directory = Path.cwd().resolve()
repository_candidates = []
for candidate in [working_directory, *working_directory.parents]:
    repository_candidates.extend([candidate, candidate / "24H_Insights"])
REPO_ROOT = next(
    (candidate for candidate in repository_candidates if (candidate / "VideoModule").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing VideoModule")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from VideoModule.detection import (
    get_detection_summary,
    load_yolo_model,
    run_yolo_detection,
    visualize_yolo_detections,
)
from VideoModule.io.clip_export import ffprobe_path

PIPELINE_ROOT = REPO_ROOT / "stoppage_detection_and_classification"
OBJECT_DETECTION_DIR = PIPELINE_ROOT / "object_detection"
PLC_EVENTS_DIR = PIPELINE_ROOT / "plc_stoppage_events"
PLC_EVENTS_OUTPUT_DIR = PLC_EVENTS_DIR / "output" / "01_prepare_events"
PLC_L14_EVENTS_DIR = PLC_EVENTS_DIR / "data" / "plc_l14_events"

DEFAULT_CHECKPOINT = OBJECT_DETECTION_DIR / "models" / "yolov8n_paired_best.pt"

@dataclass(frozen=True)
class Config:
    checkpoint: Path = DEFAULT_CHECKPOINT
    output_dir: Path = OBJECT_DETECTION_DIR / "output" / "02_detect_s3_clips"
    source_events_xls: Path = PLC_L14_EVENTS_DIR / "L14 Shrinkwrap - Unhealth Events FYTD.xls"
    aws_profile: str = "DashcamGlbDiageoProdDataContrib-522196013725"
    s3_bucket: str = "diageo-prod-global-dashcam-mc-nuc-video"
    s3_prefix: str = "cortexvpu-01a-005-41884872/"
    start_timestamp_utc: str | None = "2026-07-02 00:00:00"
    end_timestamp_utc: str | None = None
    events_csv: Path | None = None  # None selects the newest included extract.
    event_timezone: str = "Europe/London"
    event_window_before_seconds: float = 10.0
    event_window_after_seconds: float = 5.0
    listing_padding_seconds: float = 30.0
    duration_probe_timeout_seconds: float = 120.0
    sample_fps: float | None = None  # None processes every frame.
    confidence_threshold: float = 0.25
    fallen_detection_threshold: float = 0.55
    iou_threshold: float = 0.45
    rotate_from_timestamp_utc: datetime = datetime(2026, 6, 21)
    minimum_fallen_frames: int = 3
    save_annotated_videos: bool = True
    run_s3_batch: bool = True
    max_clips: int | None = None
    reprocess_successful: bool = False

CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)
(CFG.output_dir / "downloads").mkdir(parents=True, exist_ok=True)
(CFG.output_dir / "annotated_videos").mkdir(parents=True, exist_ok=True)
(CFG.output_dir / "annotated_peak_frames" / "fallen_detected").mkdir(parents=True, exist_ok=True)

if not CFG.checkpoint.is_file():
    raise FileNotFoundError(f"Detector checkpoint not found: {CFG.checkpoint}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required; select the 24H Object Detection (CUDA) kernel")
if CFG.sample_fps is not None and CFG.sample_fps <= 0:
    raise ValueError("sample_fps must be None or positive")
if not 0 <= CFG.confidence_threshold <= 1:
    raise ValueError("confidence_threshold must be between 0 and 1")
if not 0 <= CFG.fallen_detection_threshold <= 1:
    raise ValueError("fallen_detection_threshold must be between 0 and 1")

print(f"Checkpoint: {CFG.checkpoint}")
print(f"Output:     {CFG.output_dir}")
print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"Sampling:   {'every frame' if CFG.sample_fps is None else f'{CFG.sample_fps:g} FPS'}")
print(
    f"PLC window: {CFG.event_window_before_seconds:g}s before to "
    f"{CFG.event_window_after_seconds:g}s after spreadsheet time"
)
print(f"Fallen threshold: {CFG.fallen_detection_threshold:.2f}")
print(f"Run batch:  {CFG.run_s3_batch}")

Checkpoint: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\object_detection\data\object_detection_poc_runs_paired\yolo_n\weights\best.pt
Output:     C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\johnnie_walker_s3_object_detection_results
GPU:        NVIDIA GeForce RTX 5080 Laptop GPU
Sampling:   every frame
PLC window: 0.5 minute(s) each side
Run batch:  True


## 2. Load the detector on CUDA

In [12]:
detector = load_yolo_model(CFG.checkpoint, device="cuda:0")
print(f"Detector device: {detector.device}")
print(f"Detector classes: {detector.class_names}")
if not str(detector.device).startswith("cuda"):
    raise RuntimeError(f"Expected CUDA detector, got {detector.device}")

2026-07-27 16:19:23,146 - INFO - Initialized YOLODetector with model size: yolov8m
2026-07-27 16:19:23,147 - INFO -   Device: cuda
2026-07-27 16:19:23,147 - INFO - Loading YOLO checkpoint: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\object_detection\data\object_detection_poc_runs_paired\yolo_n\weights\best.pt
2026-07-27 16:19:23,167 - INFO - ✓ Loaded checkpoint with 2 classes
2026-07-27 16:19:23,168 - INFO -   Classes: ['fallen_bottle', 'upright_bottles']


Detector device: cuda:0
Detector classes: ['fallen_bottle', 'upright_bottles']


## 3. Build the PLC-event-window S3 queue

Authenticate first if needed:

```powershell
aws sso login --profile DashcamGlbDiageoProdDataContrib-522196013725
```

In [13]:
VIDEO_EXTENSIONS = {".ts", ".mp4", ".avi", ".mov", ".mkv", ".m4v"}
TIMESTAMP_PATTERN = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d{6})")

def parse_key_timestamp(key):
    match = TIMESTAMP_PATTERN.search(Path(key).name)
    return datetime.strptime(match.group(1), "%Y-%m-%d_%H-%M-%S_%f") if match else None

def optional_timestamp(value):
    if value is None:
        return None
    timestamp = pd.Timestamp(value)
    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)
    return timestamp.to_pydatetime()

def create_s3_client(profile_name):
    try:
        return boto3.Session(profile_name=profile_name).client("s3")
    except (BotoCoreError, ClientError, TokenRetrievalError) as error:
        raise RuntimeError(
            f"AWS authentication failed. Run: aws sso login --profile {profile_name}"
        ) from error

def find_latest_event_extract():
    candidates = sorted(
        PLC_EVENTS_OUTPUT_DIR.glob("johnnie_walker_black_label_events_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No Johnnie Walker event extract found in {PLC_EVENTS_OUTPUT_DIR}")
    return candidates[0]

def load_events(events_csv, event_timezone):
    events = pd.read_csv(events_csv)
    if "Event Time" not in events:
        raise ValueError("Event extract has no 'Event Time' column")
    if "Include_Status" in events:
        events = events[
            events["Include_Status"].astype(str).str.strip().str.casefold().eq("include")
        ].copy()
    if "Product" in events:
        events = events[
            events["Product"].astype(str).str.contains(
                r"(?:johnnie\s+walker|jw)\s+black", case=False, na=False, regex=True
            )
        ].copy()
    if events.empty:
        raise ValueError("Event extract contains no included Johnnie Walker events")
    local_times = pd.to_datetime(events["Event Time"], errors="raise").dt.tz_localize(
        event_timezone, ambiguous="raise", nonexistent="raise"
    )
    events = events.reset_index(drop=True)
    events.insert(0, "event_id", [f"event_{index + 1:06d}" for index in range(len(events))])
    events["event_time_local"] = local_times.reset_index(drop=True)
    events["event_time_utc"] = local_times.dt.tz_convert("UTC").reset_index(drop=True)
    return events

def successful_keys(csv_path):
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return set()
    table = pd.read_csv(csv_path)
    if "status" in table:
        table = table[table["status"].astype(str).str.casefold().eq("ok")]
    return set(table.get("s3_key", pd.Series(dtype=str)).dropna().astype(str))

def list_video_queue(s3_client, config):
    start = optional_timestamp(config.start_timestamp_utc)
    end = optional_timestamp(config.end_timestamp_utc)
    rows = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=config.s3_bucket, Prefix=config.s3_prefix):
        for item in page.get("Contents", []):
            key = str(item["Key"])
            if Path(key).suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            timestamp = parse_key_timestamp(key)
            if timestamp is None:
                continue
            if start is not None and timestamp < start:
                continue
            if end is not None and timestamp > end:
                continue
            rows.append({
                "s3_key": key,
                "filename": Path(key).name,
                "clip_timestamp_utc": timestamp,
                "size_bytes": int(item.get("Size", 0)),
            })
    queue = pd.DataFrame(rows)
    if not queue.empty:
        queue = queue.sort_values(["clip_timestamp_utc", "filename"]).reset_index(drop=True)
    return queue

def duration_candidates_for_events(events, listed_queue, before_seconds, after_seconds):
    if listed_queue.empty:
        return listed_queue.copy()
    timestamps = listed_queue["clip_timestamp_utc"].tolist()
    selected_indices = set()
    for event_time in events["event_time_utc"]:
        event_time = event_time.tz_convert("UTC").tz_localize(None).to_pydatetime()
        window_start = event_time - timedelta(seconds=before_seconds)
        window_end = event_time + timedelta(seconds=after_seconds)
        inside = [index for index, timestamp in enumerate(timestamps) if window_start <= timestamp <= window_end]
        if inside:
            selected_indices.update(inside)
            if min(inside) > 0:
                selected_indices.add(min(inside) - 1)
        else:
            preceding = [index for index, timestamp in enumerate(timestamps) if timestamp < window_start]
            if preceding:
                selected_indices.add(preceding[-1])
    return listed_queue.iloc[sorted(selected_indices)].reset_index(drop=True)

def resolve_ffprobe():
    executable = ffprobe_path()
    if executable is not None:
        return Path(executable)
    local_app_data = Path(os.environ.get("LOCALAPPDATA", ""))
    winget_packages = local_app_data / "Microsoft" / "WinGet" / "Packages"
    candidates = sorted(
        winget_packages.glob("Gyan.FFmpeg*/ffmpeg-*/bin/ffprobe.exe"),
        reverse=True,
    )
    if candidates:
        return candidates[0]
    raise RuntimeError("ffprobe is required to match clips to PLC event windows")

def probe_duration_seconds(s3_client, bucket, key, timeout_seconds):
    executable = resolve_ffprobe()
    url = s3_client.generate_presigned_url(
        "get_object", Params={"Bucket": bucket, "Key": key}, ExpiresIn=3600
    )
    completed = subprocess.run(
        [str(executable), "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", url],
        capture_output=True, text=True, timeout=timeout_seconds, check=True,
    )
    duration = float(completed.stdout.strip())
    if duration <= 0:
        raise ValueError(f"Invalid duration for {key}: {duration}")
    return duration

def load_or_probe_durations(s3_client, candidates, cache_path, config):
    cached = {}
    if cache_path.exists() and cache_path.stat().st_size:
        cache = pd.read_csv(cache_path)
        cached = {
            str(row.s3_key): float(row.duration_seconds)
            for row in cache.itertuples()
            if str(getattr(row, "status", "ok")) == "ok" and pd.notna(row.duration_seconds)
        }
    rows = []
    for item in candidates.itertuples():
        if item.s3_key in cached:
            continue
        try:
            duration = probe_duration_seconds(
                s3_client, config.s3_bucket, item.s3_key, config.duration_probe_timeout_seconds
            )
            cached[item.s3_key] = duration
            rows.append({"s3_key": item.s3_key, "duration_seconds": duration, "status": "ok", "error": ""})
        except Exception as error:
            rows.append({"s3_key": item.s3_key, "duration_seconds": "", "status": "failed", "error": str(error)})
    if rows:
        append_header = not cache_path.exists() or cache_path.stat().st_size == 0
        pd.DataFrame(rows).to_csv(cache_path, mode="a", header=append_header, index=False)
    return cached

def build_event_window_allowlist(events, candidates, duration_by_key, before_seconds, after_seconds):
    rows = []
    for event in events.itertuples():
        event_time = event.event_time_utc.tz_convert("UTC").tz_localize(None).to_pydatetime()
        window_start = event_time - timedelta(seconds=before_seconds)
        window_end = event_time + timedelta(seconds=after_seconds)
        for clip in candidates.itertuples():
            duration = duration_by_key.get(clip.s3_key)
            if duration is None:
                continue
            clip_end = clip.clip_timestamp_utc + timedelta(seconds=duration)
            if clip.clip_timestamp_utc <= window_end and clip_end >= window_start:
                rows.append({
                    "event_id": event.event_id,
                    "event_time_utc": event_time.isoformat(),
                    "window_start_utc": window_start.isoformat(),
                    "window_end_utc": window_end.isoformat(),
                    "s3_key": clip.s3_key,
                    "clip_timestamp_utc": clip.clip_timestamp_utc.isoformat(),
                    "duration_seconds": duration,
                })
    return pd.DataFrame(rows)

s3_client = create_s3_client(CFG.aws_profile)
clip_results_path = CFG.output_dir / "clip_detection_results.csv"
frame_results_path = CFG.output_dir / "frame_detection_results.csv"
listed_queue = list_video_queue(s3_client, CFG)
events_csv = Path(CFG.events_csv or find_latest_event_extract()).resolve()
events = load_events(events_csv, CFG.event_timezone)
duration_candidates = duration_candidates_for_events(
    events,
    listed_queue,
    CFG.event_window_before_seconds,
    CFG.event_window_after_seconds,
)
duration_cache_path = CFG.output_dir / "s3_clip_duration_cache.csv"
duration_by_key = load_or_probe_durations(
    s3_client, duration_candidates, duration_cache_path, CFG
)
event_window_clips = build_event_window_allowlist(
    events,
    duration_candidates,
    duration_by_key,
    CFG.event_window_before_seconds,
    CFG.event_window_after_seconds,
)
event_window_clips.to_csv(CFG.output_dir / "event_window_clips.csv", index=False)
allowed_keys = set(event_window_clips.get("s3_key", pd.Series(dtype=str)).astype(str))
queue = listed_queue[listed_queue["s3_key"].isin(allowed_keys)].reset_index(drop=True)
completed = successful_keys(clip_results_path)
if not CFG.reprocess_successful and not queue.empty:
    queue = queue[~queue["s3_key"].isin(completed)].reset_index(drop=True)
if CFG.max_clips is not None:
    queue = queue.head(CFG.max_clips).reset_index(drop=True)

print(f"Events extract:       {events_csv}")
print(f"Included events:      {len(events)}")
print(
    f"Event window:         -{CFG.event_window_before_seconds:g}s / "
    f"+{CFG.event_window_after_seconds:g}s"
)
print(f"Allowlisted S3 clips: {len(allowed_keys)}")
print(f"Previously completed: {len(completed)}")
print(f"Queued clips:         {len(queue)}")
display(queue.head(20))

2026-07-27 16:19:23,322 - INFO - Loading cached SSO token for DashcamGlbDiageoProdDataContrib-522196013725


Events extract:       C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\01_prepare_events\johnnie_walker_black_label_events_2026-07-23.csv
Included events:      251
Event window:         +/- 0.5 minute(s)
Allowlisted S3 clips: 0
Previously completed: 0
Queued clips:         0


,s3_key,filename,clip_timestamp_utc,size_bytes


## 4. Detection and resumable CSV helpers

In [14]:
def append_rows(path, rows):
    if not rows:
        return
    exists = path.exists() and path.stat().st_size > 0
    columns = list(rows[0])
    if exists:
        existing_columns = list(pd.read_csv(path, nrows=0).columns)
        if existing_columns != columns:
            raise ValueError(f"CSV schema mismatch for {path.name}")
    pd.DataFrame(rows, columns=columns).to_csv(
        path, mode="a", header=not exists, index=False
    )

def detection_payload(result):
    return [
        {
            "class_id": int(item.class_id),
            "class_name": str(item.class_name),
            "confidence": float(item.confidence),
            "box_xyxy": [float(value) for value in item.box],
        }
        for item in result.detections
    ]

def detect_video(video_path, annotated_path, clip_timestamp, config):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    source_fps = float(capture.get(cv2.CAP_PROP_FPS))
    if source_fps <= 0:
        source_fps = 30.0
    sample_stride = 1 if config.sample_fps is None else max(1, round(source_fps / config.sample_fps))
    output_fps = source_fps / sample_stride
    rotate_180 = clip_timestamp >= config.rotate_from_timestamp_utc
    writer = None
    rows = []
    decoded_count = 0
    processed_count = 0
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            frame_index = decoded_count
            decoded_count += 1
            if frame_index % sample_stride != 0:
                continue
            if rotate_180:
                frame = cv2.rotate(frame, cv2.ROTATE_180)
            result = run_yolo_detection(
                detector,
                frame,
                confidence_threshold=config.confidence_threshold,
                iou_threshold=config.iou_threshold,
                device="cuda:0",
            )
            summary = get_detection_summary(result)
            payload = detection_payload(result)
            fallen_scores = [x["confidence"] for x in payload if x["class_name"] == "fallen_bottle"]
            upright_scores = [x["confidence"] for x in payload if x["class_name"] == "upright_bottles"]
            frame_time = frame_index / source_fps
            rows.append({
                "frame_index": frame_index,
                "frame_time_seconds": frame_time,
                "estimated_frame_timestamp_utc": (pd.Timestamp(clip_timestamp) + pd.to_timedelta(frame_time, unit="s")).isoformat(),
                "detection_count": len(payload),
                "upright_bottles_count": int(summary.get("upright_bottles", 0)),
                "fallen_bottle_count": int(summary.get("fallen_bottle", 0)),
                "maximum_upright_confidence": max(upright_scores, default=0.0),
                "maximum_fallen_confidence": max(fallen_scores, default=0.0),
                "detections_json": json.dumps(payload, separators=(",", ":")),
                "inference_ms": float(result.timing_ms.get("total", 0.0)),
            })
            processed_count += 1
            if config.save_annotated_videos:
                rendered = visualize_yolo_detections(frame, result)
                if writer is None:
                    height, width = rendered.shape[:2]
                    writer = cv2.VideoWriter(
                        str(annotated_path),
                        cv2.VideoWriter_fourcc(*"mp4v"),
                        output_fps,
                        (width, height),
                    )
                    if not writer.isOpened():
                        raise RuntimeError(f"Could not open video writer: {annotated_path}")
                writer.write(rendered)
    finally:
        capture.release()
        if writer is not None:
            writer.release()
    return rows, {
        "decoded_frame_count": decoded_count,
        "processed_frame_count": processed_count,
        "source_fps": source_fps,
        "effective_sample_fps": output_fps,
        "rotated_180": rotate_180,
    }

def summarize_clip(frame_rows, minimum_fallen_frames, fallen_threshold):
    fallen_rows = [
        row
        for row in frame_rows
        if row["fallen_bottle_count"] > 0
        and row["maximum_fallen_confidence"] >= fallen_threshold
    ]
    upright_rows = [row for row in frame_rows if row["upright_bottles_count"] > 0]
    if len(fallen_rows) >= minimum_fallen_frames:
        prediction = "fallen_detected"
        confidence = max(row["maximum_fallen_confidence"] for row in fallen_rows)
    elif upright_rows:
        prediction = "upright_group_detected"
        confidence = max(row["maximum_upright_confidence"] for row in upright_rows)
    else:
        prediction = "no_detection"
        confidence = 0.0
    return prediction, confidence, fallen_rows, upright_rows

def save_annotated_peak_fallen_frame(video_path, peak_row, rotate_180, destination_path, config):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not reopen video for peak frame: {video_path}")
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(peak_row["frame_index"]))
        ok, frame = capture.read()
    finally:
        capture.release()
    if not ok:
        raise RuntimeError(f"Could not read peak frame {peak_row['frame_index']}")
    if rotate_180:
        frame = cv2.rotate(frame, cv2.ROTATE_180)
    result = run_yolo_detection(
        detector,
        frame,
        confidence_threshold=config.confidence_threshold,
        iou_threshold=config.iou_threshold,
        device="cuda:0",
    )
    rendered = visualize_yolo_detections(frame, result)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(destination_path), rendered):
        raise RuntimeError(f"Could not save peak frame: {destination_path}")
    return destination_path

## 5. Run the S3 object-detection batch

Set `run_s3_batch=True` in the configuration cell after reviewing the queue.

In [15]:
if not CFG.run_s3_batch:
    print("S3 detection is disabled. Set CFG.run_s3_batch=True and rerun from the configuration cell.")
else:
    for queue_index, item in queue.iterrows():
        s3_key = str(item["s3_key"])
        filename = str(item["filename"])
        clip_timestamp = pd.Timestamp(item["clip_timestamp_utc"]).to_pydatetime()
        key_hash = hashlib.sha1(s3_key.encode("utf-8")).hexdigest()[:10]
        artifact_stem = f"{clip_timestamp.strftime('%Y-%m-%d_%H-%M-%S_%f')}__{key_hash}"
        source_suffix = Path(filename).suffix.lower()
        temporary_path = CFG.output_dir / "downloads" / f"{Path(filename).stem}__{key_hash}{source_suffix}"
        partial_path = temporary_path.with_suffix(temporary_path.suffix + ".part")
        pending_annotated_path = CFG.output_dir / "annotated_videos" / f"{artifact_stem}.pending.mp4"
        final_annotated_path = None
        print(f"[{queue_index + 1}/{len(queue)}] {filename}")
        try:
            s3_client.download_file(CFG.s3_bucket, s3_key, str(partial_path))
            partial_path.replace(temporary_path)
            frame_rows, video_stats = detect_video(
                temporary_path, pending_annotated_path, clip_timestamp, CFG
            )
            prediction, confidence, fallen_rows, upright_rows = summarize_clip(
                frame_rows,
                CFG.minimum_fallen_frames,
                CFG.fallen_detection_threshold,
            )
            peak_frame_path = ""
            peak_fallen_frame_index = ""
            peak_fallen_confidence = ""
            if prediction == "fallen_detected":
                peak_row = max(fallen_rows, key=lambda row: row["maximum_fallen_confidence"])
                peak_fallen_frame_index = int(peak_row["frame_index"])
                peak_fallen_confidence = float(peak_row["maximum_fallen_confidence"])
                peak_destination = (
                    CFG.output_dir
                    / "annotated_peak_frames"
                    / "fallen_detected"
                    / f"{artifact_stem}__frame_{peak_fallen_frame_index:06d}.jpg"
                )
                peak_frame_path = str(save_annotated_peak_fallen_frame(
                    temporary_path,
                    peak_row,
                    video_stats["rotated_180"],
                    peak_destination,
                    CFG,
                ))
            if CFG.save_annotated_videos:
                result_video_dir = CFG.output_dir / "annotated_videos" / prediction
                result_video_dir.mkdir(parents=True, exist_ok=True)
                final_annotated_path = result_video_dir / f"{artifact_stem}_detected.mp4"
                pending_annotated_path.replace(final_annotated_path)
            for row in frame_rows:
                row.update({
                    "s3_bucket": CFG.s3_bucket,
                    "s3_key": s3_key,
                    "video_name": filename,
                    "clip_timestamp_utc": clip_timestamp.isoformat(),
                })
            append_rows(frame_results_path, frame_rows)
            append_rows(clip_results_path, [{
                "processed_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                "status": "ok",
                "error": "",
                "s3_bucket": CFG.s3_bucket,
                "s3_key": s3_key,
                "video_name": filename,
                "clip_timestamp_utc": clip_timestamp.isoformat(),
                "checkpoint": str(CFG.checkpoint),
                "prediction": prediction,
                "confidence": confidence,
                "fallen_detection_threshold": CFG.fallen_detection_threshold,
                "frames_with_fallen": len(fallen_rows),
                "frames_with_upright_group": len(upright_rows),
                "first_fallen_frame_index": fallen_rows[0]["frame_index"] if fallen_rows else "",
                "first_fallen_time_seconds": fallen_rows[0]["frame_time_seconds"] if fallen_rows else "",
                "peak_fallen_frame_index": peak_fallen_frame_index,
                "peak_fallen_confidence": peak_fallen_confidence,
                "annotated_peak_frame_path": peak_frame_path,
                "annotated_video_path": str(final_annotated_path) if final_annotated_path else "",
                **video_stats,
            }])
            print(f"  {prediction} ({confidence:.3f}); processed {video_stats['processed_frame_count']} frames")
        except Exception as error:
            append_rows(clip_results_path, [{
                "processed_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                "status": "failed",
                "error": str(error),
                "s3_bucket": CFG.s3_bucket,
                "s3_key": s3_key,
                "video_name": filename,
                "clip_timestamp_utc": clip_timestamp.isoformat(),
                "checkpoint": str(CFG.checkpoint),
                "prediction": "",
                "confidence": "",
                "fallen_detection_threshold": CFG.fallen_detection_threshold,
                "frames_with_fallen": "",
                "frames_with_upright_group": "",
                "first_fallen_frame_index": "",
                "first_fallen_time_seconds": "",
                "peak_fallen_frame_index": "",
                "peak_fallen_confidence": "",
                "annotated_peak_frame_path": "",
                "annotated_video_path": "",
                "decoded_frame_count": "",
                "processed_frame_count": "",
                "source_fps": "",
                "effective_sample_fps": "",
                "rotated_180": clip_timestamp >= CFG.rotate_from_timestamp_utc,
            }])
            print(f"  FAILED: {error}")
        finally:
            partial_path.unlink(missing_ok=True)
            temporary_path.unlink(missing_ok=True)
            pending_annotated_path.unlink(missing_ok=True)

## 6. Review results

In [18]:
if clip_results_path.exists():
    clip_results = pd.read_csv(clip_results_path)
    print(clip_results["status"].value_counts(dropna=False).to_string())
    if "prediction" in clip_results:
        print()
        print(clip_results.loc[clip_results["status"].eq("ok"), "prediction"].value_counts().to_string())
    display(clip_results.tail(30))
else:
    print("No clip results exist yet.")

status
ok    86

prediction
upright_group_detected    45
no_detection              29
fallen_detected           12


,processed_at_utc,status,error,s3_bucket,s3_key,video_name,clip_timestamp_utc,checkpoint,prediction,confidence,...,first_fallen_time_seconds,peak_fallen_frame_index,peak_fallen_confidence,annotated_peak_frame_path,annotated_video_path,decoded_frame_count,processed_frame_count,source_fps,effective_sample_fps,rotated_180
56,2026-07-27T17:19:55.987900+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-27-02...,2026-07-07T11:27:02.116667,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,483,483,60.0,60.0,True
57,2026-07-27T17:20:54.797354+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-27-08...,2026-07-07T11:27:08.116667,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2105,2105,60.0,60.0,True
58,2026-07-27T17:21:10.182044+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-27-40...,2026-07-07T11:27:40.116667,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,545,545,60.0,60.0,True
59,2026-07-27T17:21:21.055041+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-27-47...,2026-07-07T11:27:47.116667,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,365,365,60.0,60.0,True
60,2026-07-27T17:21:42.529462+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-37-14...,2026-07-07T11:37:14.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,738,738,60.0,60.0,True
61,2026-07-27T17:22:17.347662+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_11-37-24...,2026-07-07T11:37:24.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,1225,1225,60.0,60.0,True
62,2026-07-27T17:22:33.292304+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_12-28-13...,2026-07-07T12:28:13.983333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,upright_group_detected,0.966734,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,549,549,60.0,60.0,True
63,2026-07-27T17:22:59.383696+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_12-28-20...,2026-07-07T12:28:20.983333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,upright_group_detected,0.963017,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,909,909,60.0,60.0,True
64,2026-07-27T17:23:08.514026+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_12-28-33...,2026-07-07T12:28:33.983333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,no_detection,0.000000,...,NaN,NaN,NaN,NaN,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,310,310,60.0,60.0,True
65,2026-07-27T17:23:24.786836+00:00,ok,NaN,diageo-prod-global-dashcam-mc-nuc-video,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-07_12-29-03...,2026-07-07T12:29:03.983333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,upright_group_detect

## 7. Distinguish fallen before entry from fallen in view

This runs entirely on the saved frame detections, so changing the temporal rules does not require GPU inference again.

In [19]:
temporal_results_path = CFG.output_dir / "event_temporal_classification.csv"

if not frame_results_path.exists() or frame_results_path.stat().st_size == 0:
    event_temporal_results = pd.DataFrame()
    print("No frame detections exist yet. Run the S3 batch first.")
else:
    frame_results = pd.read_csv(frame_results_path)
    event_frames = (
        event_window_clips[["event_id", "s3_key"]]
        .drop_duplicates()
        .merge(frame_results, on="s3_key", how="inner")
    )
    event_frames["estimated_frame_timestamp_utc"] = pd.to_datetime(
        event_frames["estimated_frame_timestamp_utc"], format="mixed", utc=True
    )

    temporal_rows = []
    for event_id, frames in event_frames.groupby("event_id", sort=False):
        frames = frames.sort_values("estimated_frame_timestamp_utc").reset_index(drop=True)
        fallen = frames["maximum_fallen_confidence"].fillna(0).ge(0.55)
        upright = frames["maximum_upright_confidence"].fillna(0).ge(0.40)
        stable_fallen_end = fallen.astype(int).rolling(3).sum().eq(3)
        stable_positions = stable_fallen_end[stable_fallen_end].index.tolist()

        classification = "no_fall_detected"
        evidence_video = ""
        first_fallen_timestamp = ""
        first_fallen_confidence = ""

        if stable_positions:
            first_fallen_position = stable_positions[0] - 2
            first_fallen_row = frames.iloc[first_fallen_position]
            seconds_from_first_frame = (
                first_fallen_row["estimated_frame_timestamp_utc"]
                - frames.iloc[0]["estimated_frame_timestamp_utc"]
            ).total_seconds()
            upright_seen_before = bool(upright.iloc[:first_fallen_position].any())

            if upright_seen_before:
                classification = "fallen_in_view"
            elif seconds_from_first_frame <= 1.0:
                classification = "fallen_before_entry"
            else:
                classification = "uncertain"

            evidence_video = str(first_fallen_row["video_name"])
            first_fallen_timestamp = first_fallen_row["estimated_frame_timestamp_utc"].isoformat()
            first_fallen_confidence = float(first_fallen_row["maximum_fallen_confidence"])

        temporal_rows.append({
            "event_id": event_id,
            "classification": classification,
            "video_name": evidence_video,
            "first_fallen_timestamp_utc": first_fallen_timestamp,
            "first_fallen_confidence": first_fallen_confidence,
            "frame_count": len(frames),
        })

    event_temporal_results = pd.DataFrame(temporal_rows)
    event_temporal_results.to_csv(temporal_results_path, index=False)
    print(event_temporal_results["classification"].value_counts().to_string())
    print(f"Saved: {temporal_results_path}")
    display(event_temporal_results.head(30))

classification
no_fall_detected    32
fallen_in_view      12
uncertain            1
Saved: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\object_detection\output\02_detect_s3_clips\event_temporal_classification.csv


,event_id,classification,video_name,first_fallen_timestamp_utc,first_fallen_confidence,frame_count
0,event_000001,no_fall_detected,,,,4058
1,event_000002,no_fall_detected,,,,1975
2,event_000003,fallen_in_view,cortexvpu-01a-005-41884872_2026-07-07_14-03-52...,2026-07-07T14:03:54.883333667+00:00,0.776506,1930
3,event_000004,no_fall_detected,,,,1820
4,event_000005,fallen_in_view,cortexvpu-01a-005-41884872_2026-07-07_13-25-27...,2026-07-07T13:25:36.833333333+00:00,0.618405,1235
5,event_000006,no_fall_detected,,,,671
6,event_000007,no_fall_detected,,,,1438
7,event_000008,no_fall_detected,,,,1065
8,event_000009,no_fall_detected,,,,1762
9,event_000010,no_fall_detected,,,,1768


## 8. Export the L14 incident CSV

This reads the original `L14 Shrinkwrap - Unhealth Events FYTD.xls` table and preserves its event columns and row order. It appends `Classification of Incident` and `Video File Name Showing Incident`. A `fallen_detected` clip is preferred; otherwise the highest-confidence successful result in that event's 10-second-before/5-second-after window is used.

In [20]:
INCIDENT_CLASSIFICATION_COLUMN = "Classification of Incident"
INCIDENT_VIDEO_COLUMN = "Video File Name Showing Incident"
incident_export_path = CFG.output_dir / "L14 Shrinkwrap - Unhealth Events FYTD_object_detection.csv"

def export_l14_incident_csv(source_xls, events, event_window_clips, clip_results_path, output_path):
    source_xls = Path(source_xls)
    if not source_xls.is_file():
        raise FileNotFoundError(f"Source L14 spreadsheet not found: {source_xls}")

    # Row 4 of the XLS contains the real table headers; the first three rows are report titles.
    output = pd.read_excel(source_xls, skiprows=3)
    output[INCIDENT_CLASSIFICATION_COLUMN] = ""
    output[INCIDENT_VIDEO_COLUMN] = ""

    if not clip_results_path.exists() or clip_results_path.stat().st_size == 0:
        output.to_csv(output_path, index=False)
        return output

    clip_results = pd.read_csv(clip_results_path)
    successful = clip_results[clip_results["status"].astype(str).str.casefold().eq("ok")].copy()
    if successful.empty or event_window_clips.empty:
        output.to_csv(output_path, index=False)
        return output

    # Use the newest successful result for each S3 object, then attach clips to PLC events.
    successful = successful.drop_duplicates("s3_key", keep="last")
    event_clip_results = event_window_clips[["event_id", "s3_key"]].merge(
        successful,
        on="s3_key",
        how="inner",
    )
    if event_clip_results.empty:
        output.to_csv(output_path, index=False)
        return output

    # Fallen evidence takes priority; within a class choose the strongest confidence.
    prediction_priority = {
        "fallen_detected": 0,
        "upright_group_detected": 1,
        "no_detection": 2,
    }
    event_clip_results["prediction_priority"] = (
        event_clip_results["prediction"].map(prediction_priority).fillna(99)
    )
    event_clip_results["confidence_numeric"] = pd.to_numeric(
        event_clip_results["confidence"], errors="coerce"
    ).fillna(-1.0)
    selected = (
        event_clip_results
        .sort_values(
            ["event_id", "prediction_priority", "confidence_numeric"],
            ascending=[True, True, False],
        )
        .drop_duplicates("event_id", keep="first")
        .set_index("event_id")
    )

    # Prefer the event-level temporal decision produced in the preceding cell.
    if "event_temporal_results" in globals() and not event_temporal_results.empty:
        selected = (
            event_temporal_results[["event_id", "classification", "video_name"]]
            .rename(columns={"classification": "prediction"})
            .set_index("event_id")
        )

    # Match the filtered event records back to the original XLS using exact Event Time.
    source_event_column = next(
        column
        for column in output.columns
        if str(column).replace("\n", " ").strip().casefold() == "event time"
    )
    event_id_by_time = {
        pd.Timestamp(event_time): event_id
        for event_id, event_time in zip(events["event_id"], events["Event Time"])
    }

    for row_index, event_time in output[source_event_column].items():
        if pd.isna(event_time):
            continue
        event_id = event_id_by_time.get(pd.Timestamp(event_time))
        if event_id is None or event_id not in selected.index:
            continue
        result = selected.loc[event_id]
        output.at[row_index, INCIDENT_CLASSIFICATION_COLUMN] = str(result["prediction"])
        output.at[row_index, INCIDENT_VIDEO_COLUMN] = str(result["video_name"])

    output.to_csv(output_path, index=False)
    return output

incident_export = export_l14_incident_csv(
    source_xls=CFG.source_events_xls,
    events=events,
    event_window_clips=event_window_clips,
    clip_results_path=clip_results_path,
    output_path=incident_export_path,
)

classified_incidents = incident_export[
    incident_export[INCIDENT_CLASSIFICATION_COLUMN].astype(str).str.len() > 0
]
print(f"L14 rows exported:       {len(incident_export):,}")
print(f"Rows with classification: {len(classified_incidents):,}")
print(f"Saved: {incident_export_path}")
display(classified_incidents[[INCIDENT_CLASSIFICATION_COLUMN, INCIDENT_VIDEO_COLUMN]].head(30))

L14 rows exported:       1,480
Rows with classification: 45
Saved: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\object_detection\output\02_detect_s3_clips\L14 Shrinkwrap - Unhealth Events FYTD_object_detection.csv


,Classification of Incident,Video File Name Showing Incident
282,no_fall_detected,
287,no_fall_detected,
291,fallen_in_view,cortexvpu-01a-005-41884872_2026-07-07_14-03-52...
293,no_fall_detected,
295,fallen_in_view,cortexvpu-01a-005-41884872_2026-07-07_13-25-27...
297,no_fall_detected,
299,no_fall_detected,
319,no_fall_detected,
321,no_fall_detected,
323,no_fall_detected,
